In [1]:
from qiskit import QuantumCircuit

n = 2  # feature dimension = 2^n
qc = QuantumCircuit(1 + 2*n, 1)

anc = 0
x = [1, 2]
y = [3, 4]

# Assume |x⟩ and |y⟩ already prepared

qc.h(anc)

for i in range(n):
    qc.cswap(anc, 1+i, 1+n+i)

qc.h(anc)
qc.measure(anc, 0)


In [14]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.compiler import transpile

def swap_test_euclidean_distance(x, y, n, shots=1000):
    # Qubits: 1 ancilla + n (x) + n (y)
    qc = QuantumCircuit(1 + 2*n, 1)

    ancilla = 0
    x_reg = list(range(1, 1 + n))
    y_reg = list(range(1 + n, 1 + 2*n))

    # Prepare |x⟩ and |y⟩
    qc.initialize(x, x_reg)
    qc.initialize(y, y_reg)

        # SWAP test gates (circuit)
    qc.h(ancilla)
    for i in range(n):
        qc.cswap(ancilla, x_reg[i], y_reg[i])
    qc.h(ancilla)

    # Measure ancilla
    qc.measure(ancilla, 0)

        # Simulate O(shots*logN) = O(logN)
    simulator = AerSimulator()
    tqc = transpile(qc, simulator)
    result = simulator.run(tqc, shots=shots).result()
    counts = result.get_counts()
    # Probability of measuring |0>
    p0 = counts.get("0", 0) / shots


    # O(1) classical computation
    # Fidelity |⟨x|y⟩|^2
    fidelity = max(0.0, 2 * p0 - 1)

    # Euclidean distance (normalized vectors)
    distance = np.sqrt(2 - 2 * np.sqrt(fidelity))

    return distance, fidelity


In [15]:
n = 2
x = [4, 2, 3, 1]
y = [4, 3, 2, 1]



def normalize(v):
    v = np.array(v, dtype=float)
    return v / np.linalg.norm(v)
x_n = normalize(x)
y_n = normalize(y)

q_distance, q_fidelity = swap_test_euclidean_distance(x_n, y_n, n)

print("Quantum fidelity:", q_fidelity)
print("Quantum Euclidean distance:", q_distance)

# O(N) to compute dot product (inner product) and sqrt it.
classical_distance = np.linalg.norm(x_n - y_n)

print("Classical Euclidean distance:", classical_distance)


Quantum fidelity: 0.9319999999999999
Quantum Euclidean distance: 0.2630533372083558
Classical Euclidean distance: 0.2581988897471611
